# Day 9 — LLM APIs & Prompt Engineering
Calling large language models from Python and mastering prompt engineering techniques.

In [5]:
#!pip install groq python-dotenv
#!pip install python-dotenv

In [6]:
import os
#----

In [7]:
os.environ["GROQ_API_KEY"] = ""

In [8]:

key = os.getenv("GROQ_API_KEY")

print("KEY LENGTH:", len(key))
print("KEY START:", key[:5])

KEY LENGTH: 56
KEY START: gsk_1


In [9]:
from dotenv import load_dotenv
import os

load_dotenv(".env")  # loads .env file automatically

# ── GROQ (free) ───────────────────────────────
from groq import Groq
client = Groq(api_key=os.getenv("GROQ_API_KEY"))
MODEL="llama-3.1-8b-instant"
response=client.chat.completions.create(model=MODEL, 
                                        messages=[{"role":"user", "content": "Say hello and confirm you are working."}])
print("✅ API connected!")
print(response.choices[0].message.content)

✅ API connected!
Hello, I'm functioning properly and ready to assist you with any questions or topics you'd like to discuss. What can I help you with today?


## ML Tutor Chatbot
Building a chatbot with conversation history using the system/user/assistant message structure. The system prompt defines the chatbot's personality and expertise.

In [10]:
def chat(messages):
    response=client.chat.completions.create(model=MODEL, messages=messages, temperature=0.7, max_tokens=500)
    return response.choices[0].message.content
    
# System prompt — defines chatbot personality
SYSTEM_PROMPT = """You are an expert ML tutor with 13 years of
university teaching experience. You explain machine learning
concepts clearly using simple analogies and practical examples.
Keep answers concise — under 150 words unless asked for more."""

print("Helper function ready ✓")

Helper function ready ✓


In [11]:
conversation=[{"role": "system", "content": SYSTEM_PROMPT}]
def ask_tutor(question):
    conversation.append({"role": "user", "content": question})
    answer=chat(conversation)
    conversation.append({"role": "assistant", "content":answer})
    return(answer)

# Test with 3 ML questions
questions = [
    "What is overfitting in machine learning?",
    "How does dropout help with the problem you just described?",
    "Give me a simple analogy for what we just discussed."
]
for q in questions:
    print(f"\n student: {q}")
    print(f"\n Tutor: {ask_tutor(q)}")
    print("-"*60)

print(f"\nConversation turns: {len(conversation)}")


 student: What is overfitting in machine learning?

 Tutor: Imagine you're trying to recognize your friends from a group of people you've never seen before. You start by memorizing the faces of your friends. However, if you've only seen them in a few photos, you might end up recognizing them only in those specific photos and not in new situations.

In machine learning, this is what happens when a model becomes too specialized to the training data and fails to generalize well to new, unseen data. This is called overfitting. The model has learned the noise in the training data, rather than the underlying patterns, and performs poorly on new data.

To avoid overfitting, we can use techniques like regularization, early stopping, and cross-validation. These methods help the model to be more generalizable and less dependent on the specific training data.
------------------------------------------------------------

 student: How does dropout help with the problem you just described?

 Tutor

## Prompt Engineering Techniques
The same question — different prompting styles — produces dramatically different quality answers. This is a core skill for LLM engineering.

In [12]:
# TECHNIQUE 1 — Zero-shot: ask directly, no examples
zero_shot=[{"role": "user", "content": "Classify this student feedback as Positive, Negative or Neutral: 'The course was too fast but I learned a lot'"}]

result=chat(zero_shot)
print("zero shot", result)

zero shot I would classify this student feedback as Neutral. 

The student mentions that the course was "too fast" which is a negative comment, indicating that they felt the pace was not suitable for them. However, they also mention that they "learned a lot" which is a positive comment, indicating that the content and quality of the course were good despite the issues with the pace. Overall, the feedback is mixed, hence Neutral.


In [13]:
# TECHNIQUE 2 — Few-shot: give examples before asking
few_shot=[{"role": "user", "content" : """
Classify student feedback. Examples:
"I loved the hands-on coding" → Positive
"The lectures were boring and slow" → Negative
"The course was average" → Neutral

Now classify: "The course was too fast but I learned a lot"
Answer with only one word: Positive, Negative, or Neutral.
"""}]
result = chat(few_shot)
print("FEW-SHOT:", result)

FEW-SHOT: Positive


In [14]:
# TECHNIQUE 3 — Chain-of-thought: ask model to think step by step
cot = [{"role": "user", "content": """
Classify this student feedback step by step:
"The course was too fast but I learned a lot"

Step 1: Identify negative aspects
Step 2: Identify positive aspects
Step 3: Weigh both sides
Step 4: Give final classification (Positive/Negative/Neutral)
"""}]

result = chat(cot)
print("CHAIN-OF-THOUGHT:\n", result)

CHAIN-OF-THOUGHT:
 Here's the step-by-step classification of the student feedback:

**Step 1: Identify negative aspects**
The negative aspect in the feedback is:
- "The course was too fast"

This implies that the student felt that the pace of the course was not suitable for their learning needs.

**Step 2: Identify positive aspects**
The positive aspect in the feedback is:
- "I learned a lot"

This implies that the student gained valuable knowledge and skills from the course.

**Step 3: Weigh both sides**
Both sides of the feedback are being weighed. The negative aspect (the course being too fast) might have caused some difficulty for the student, but the positive aspect (learning a lot) suggests that the student benefited from the course overall.

**Step 4: Give final classification**
Given the student's positive statement about learning a lot, the negative aspect of the course being too fast does not override the overall positive impact of the course. Therefore, the final classificat

In [15]:
# Compare results — save to DataFrame
import pandas as pd

techniques = {
    'Technique'      : ['Zero-shot', 'Few-shot', 'Chain-of-thought'],
    'Description'    : [
        'Ask directly',
        'Give 3 examples first',
        'Ask to think step by step'
    ],
    'Best for'       : [
        'Simple tasks',
        'Classification tasks',
        'Complex reasoning'
    ]
}

df_techniques = pd.DataFrame(techniques)
print(df_techniques.to_string(index=False))

       Technique               Description             Best for
       Zero-shot              Ask directly         Simple tasks
        Few-shot     Give 3 examples first Classification tasks
Chain-of-thought Ask to think step by step    Complex reasoning


## Structured Output — JSON Mode
Forcing the LLM to return JSON makes output predictable and parseable. Used in real production systems to extract structured data from unstructured text.

In [16]:
import json
def analyze_feedback_structured(feedback_text):
    messages=[{"role": "system", "content":"""you are feedback analyzer, always respond with valid JSON only. No explaination, No markdown. 
    use only this structure:
    { 
    "sentiment": "positive|Negative|Neutral",
    "score": 1-10,
    "key_issue": "main problem mentioned or null",
    "key_praise" : "main complement mentioned or null",
    "action_need": "true or fasle"}"""},
    {
    "role": "user",
    "content" : f"Analyze this feedback: {feedback_text}"
    }]
    result=chat(messages)
    try:
        return json.loads(result)
    except:
        return {"error": "JSON parse failed", "raw": result}

test_feedbacks=[
    "The neural network lecture was excellent and very clear.",
    "I could not follow the math — too fast and no examples.",
    "Average class, some parts good some parts confusing.",
    "Best course I have taken! Highly recommend this professor.",
    "The assignments had no guidance and I felt lost throughout."
]
results=[]
for fb in test_feedbacks:
    analysis=analyze_feedback_structured(fb)
    results.append(analysis)
    print(f"\n feedback: {fb[:50]}...")
    print(f"result: {json.dumps(analysis, indent=2)}")

df_structured=pd.DataFrame(results)
df_structured['feedback']=test_feedbacks
print("\n ====Structred Results============")
print(df_structured[['sentiment', 'score', 'action_need', 'feedback']].to_string(index=False))
print("\nSaved to CSV ✓")




# Save to CSV
import os
os.makedirs('data', exist_ok=True)
df_structured.to_csv('data/day9_structured_feedback.csv', index=False)
print("\nSaved to CSV ✓")



 feedback: The neural network lecture was excellent and very ...
result: {
  "sentiment": "Positive",
  "score": 9,
  "key_issue": null,
  "key_praise": "excellent and very clear",
  "action_need": "false"
}

 feedback: I could not follow the math — too fast and no exam...
result: {
  "sentiment": "Negative",
  "score": 2,
  "key_issue": "math went by too fast without examples",
  "key_praise": "null",
  "action_need": "true"
}

 feedback: Average class, some parts good some parts confusin...
result: {
  "sentiment": "Neutral",
  "score": 5,
  "key_issue": "parts confusing",
  "key_praise": "some parts good",
  "action_need": "true"
}

 feedback: Best course I have taken! Highly recommend this pr...
result: {
  "sentiment": "positive",
  "score": 10,
  "key_issue": null,
  "key_praise": "professor",
  "action_need": "false"
}

 feedback: The assignments had no guidance and I felt lost th...
result: {
  "sentiment": "Negative",
  "score": 2,
  "key_issue": "lack of guidance",
  "key_pr